# MeChess 5/5: the opening explorer by a partitioned scan (CPU notebook)

**Accelerator: None.** Only an adapter around `chessme explorer-range-scan` (also runs on a laptop). Notebook 3 reads the Lichess dump as one stream; its connection
drops about every 17 minutes and each reconnect re-reads the file from the start, so it cannot finish a month. This notebook cuts the 28 GB file into about 700 byte
windows (the file is a sequence of independent zstd frames and the server answers range requests), scans four at a time, and adds the windows up **exactly**: counts
are never sampled and never pruned inside a window.

- Counted games, bands, outputs and queries are the same as notebook 3 (see [opening-explorer.md](../docs/opening-explorer.md)).
- The scanned windows take about 45 MB each (about 31 GB): they are kept in `/kaggle/temp`, not in the 20 GB output folder. Only the merged result is output.
  A session that ends before the merge loses them, so give the run the whole 12 hours (a committed run), and expect roughly 3 to 6 hours.
- The repository comes from a dataset attached as an input (a folder that contains `chessme/`), or from `REPO_URL` when there is none.

In [ ]:
import glob, os, shutil, subprocess, sys

REPO_URL = "https://github.com/<you>/MeChess.git"      # <- put your repository URL here (not needed when the repository is attached as a dataset)
MONTH = "2026-06"                                       # a month on https://database.lichess.org (standard rated)
PARTS = 700                                             # byte windows (about 40 MB of the dump each); a rerun must use the same number
WORKERS = 4                                             # windows scanned at the same time (Kaggle CPU notebooks have 4 cores)
OUT = "/kaggle/working/explorer" if os.path.exists("/kaggle") else "explorer_local"
WINDOWS_DIR = "/kaggle/temp/windows" if os.path.exists("/kaggle") else "explorer_local/windows"

def sh(*args):
    """Run a command and stream its output; stop the notebook if it fails."""
    p = subprocess.Popen(args, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="")
    if p.wait():
        raise RuntimeError(f"failed: {' '.join(args)}")

if not os.path.exists("chessme"):
    attached = glob.glob("/kaggle/input/*/chessme/__init__.py")     # the repository attached as a dataset
    if attached:
        shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), "MeChess")
    else:                                                           # Kaggle: fetch the repository (Internet must be on)
        if "<you>" in REPO_URL:
            raise SystemExit("Attach the repository as a dataset, or set REPO_URL.")
        sh("git", "clone", "--depth", "1", REPO_URL, "MeChess")
    os.chdir("MeChess")
sh(sys.executable, "-m", "pip", "-q", "install", "python-chess", "numpy", "pyyaml", "requests", "zstandard")
CLI = [sys.executable, "-m", "chessme"]              # every step below is one `chessme` command: the same ones you run on a laptop

SOURCE = f"https://database.lichess.org/standard/lichess_db_standard_rated_{MONTH}.pgn.zst"

## Step 1. Check: read a few hundred games (seconds). Stops here if the URL, decompression or parser is broken.

In [ ]:
sh(*CLI, "explorer-build", "--check", "--source", SOURCE)

## Step 2. Scan the windows, merge them exactly and build the explorer (the long step; resumable window by window while the session lives)

In [ ]:
sh(*CLI, "explorer-range-scan", "--source", SOURCE, "--out", OUT, "--windows-dir", WINDOWS_DIR, "--parts", str(PARTS),
   "--workers", str(WORKERS), "--log", f"{OUT}/explorer_range.log")

## Step 3. Try the explorer (games, share, average rating, evaluation, and the White / draw / Black bar)

In [ ]:
AFTER_E4 = "rnbqkbnr/pppppppp/8/8/4P3/8/PPPP1PPP/RNBQKBNR b KQkq - 0 1"
for rating in (500, 900, 1500, 2000, 2500, 2800):
    print(f"--- rating {rating}, after 1.e4")
    sh(*CLI, "explorer-query", f"{OUT}/explorer.db", "--rating", str(rating), "--fen", AFTER_E4, "--top", "5")